# 10 — Four-pipeline comparison

Thin orchestration only — all logic lives in `src/reviewscope_ml/`.

Runs the four end-to-end candidates on the benchmark sample and renders the
comparison report:

| Variant | Description |
|---|---|
| `bertopic` | BERTopic off-the-shelf (MiniLM, its own UMAP+HDBSCAN; seeded) |
| `custom_hdbscan` | mpnet → UMAP(10d) → HDBSCAN (notebooks 04–06 decisions) |
| `flat_agglomerative` | mpnet → UMAP(10d) → agglomerative (ward) |
| `two_stage` | fine HDBSCAN micro-clusters → agglomerative macro merge |

**Decision protocol** (also embedded in the report): metrics shortlist 2–3
finalists; a human picks the winner from the qualitative inspection and the
intruder test. Metrics never declare the winner alone.

Runs top-to-bottom on CPU at 1k in a few minutes (embeddings are cached after
the first run). For the full 5k/50k GPU runs see `docs/methodology.md`.

In [ ]:
import sys
sys.path.insert(0, '..')  # notebooks/utils shims; package itself is pip-installed

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(message)s')

from reviewscope_ml import load_config

# CPU smoke configuration — keep this runnable on any laptop.
# GPU: use reviewscope_ml.runtime.claim_gpu() and pass device/gpu_id (see docs).
cfg = load_config(sample_size=1_000, device='cpu')
cfg.ensure_dirs()
print(cfg)

In [ ]:
# Benchmark sample (built from the raw Yelp dump if missing; idempotent)
from reviewscope_ml.data import build_benchmark_sample, subset_sample

five_k = build_benchmark_sample(cfg.project_root, 5_000)
subset_sample(five_k, cfg.data_path, cfg.sample_size)
print('benchmark:', cfg.data_path)

In [ ]:
# Run all four variants + multi-seed stability repeats, write the report.
# label_clusters=True uses Ollama when reachable and falls back to term
# labels (recorded as label_source='terms_fallback') when not.
from reviewscope_ml.eval.report import run_comparison

report_path = run_comparison(cfg, seeds=(42, 43, 44), label_clusters=True)
report_path

In [ ]:
# Render the report inline (same file a teammate reads on disk)
from IPython.display import Markdown

Markdown(report_path.read_text())

In [ ]:
# Side-by-side 2-D scatter of the four runs
import matplotlib.pyplot as plt
from reviewscope_ml.pipelines import load_run
from reviewscope_ml.pipelines.spec import VARIANTS

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, variant in zip(axes, VARIANTS):
    art = load_run(cfg.runs_dir / f'{variant}__{cfg.sample_size}__s42')
    noise = art.labels == -1
    ax.scatter(art.coords_2d[noise, 0], art.coords_2d[noise, 1], s=2, c='lightgrey', alpha=0.3)
    ax.scatter(art.coords_2d[~noise, 0], art.coords_2d[~noise, 1], s=2,
               c=art.labels[~noise], cmap='tab20', alpha=0.6)
    ax.set_title(f"{variant}\n{art.metrics['n_clusters']} clusters, "
                 f"noise {art.metrics['noise_ratio']:.0%}")
    ax.axis('off')
plt.tight_layout(); plt.show()

## Next step — the human part

1. Read the **qualitative inspection** and take the **intruder test** in the
   report above for each shortlisted finalist.
2. Open the HITL app and record your verdict:
   ```bash
   streamlit run src/reviewscope_ml/hitl/app.py
   ```
3. Only after the sign-off is recorded does the comparison have a winner.
   See `11_hitl_roundtrip.ipynb` for how recorded feedback changes the next run.